# Experimento de Reglas de Asociacion

Dataset: **Social Media User Activity Dataset**

Objetivo: limpiar y preparar el dataset como matriz binaria, ejecutar **Apriori**, **FP-Growth** y **Eclat**, comparar la calidad de sus reglas y elegir el mejor algoritmo sin usar runtime como criterio.

## 1. Configuracion sin warnings visibles

In [ ]:
import os
import warnings
import logging

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("py.warnings").setLevel(logging.ERROR)

## 2. Instalacion de dependencias

La salida se captura para evitar logs extensos en el notebook.

In [ ]:
%%capture
!pip install -q kaggle mlxtend pyECLAT

## 3. Cargar modulo del proyecto

Si abriste el notebook desde GitHub, Colab puede cargar solo el `.ipynb`. Esta celda clona el repositorio si falta la carpeta `src/`.

In [ ]:
from pathlib import Path
import sys
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

REPO_URL = "https://github.com/Shtolaa/ing_Datos_Experimento.git"
REPO_BRANCH = "dev"
REPO_DIR = Path("/content/ing_Datos_Experimento")
LOCAL_SRC = Path("../src")
COLAB_SRC = REPO_DIR / "src"

if not LOCAL_SRC.exists() and not COLAB_SRC.exists():
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
elif COLAB_SRC.exists():
    !git -C {REPO_DIR} pull -q origin {REPO_BRANCH}

PROJECT_SRC = LOCAL_SRC if LOCAL_SRC.exists() else COLAB_SRC
sys.path.insert(0, str(PROJECT_SRC.resolve()))
print(f"Modulo cargado desde: {PROJECT_SRC.resolve()}")

In [ ]:
from social_media_activity_pipeline import (
    TARGET_COLUMN,
    analyze_column_cardinality,
    choose_best_algorithm,
    clean_selected_data,
    create_binary_matrix,
    default_analysis_columns,
    discretize_numeric_columns,
    drop_columns,
    filter_happiness_rules,
    load_dataset,
    normalize_categorical_values,
    normalize_column_names,
    reduce_rare_categories,
    report_duplicates,
    report_missing_values,
    rules_to_readable,
    run_all_algorithms,
    sample_dataframe,
    split_column_types,
    suggest_columns_to_drop,
    summarize_algorithm_results,
    summarize_columns,
    validate_binary_matrix,
)

## 4. Parametros del experimento

Primero se prueba con una muestra pequena. Para la corrida final cambia `USE_SAMPLE = False` y `MAX_BINS = 10`.

In [ ]:
TARGET_COLUMN = "self_reported_happiness"
USE_SAMPLE = True
SAMPLE_SIZE = 50_000
RANDOM_STATE = 42
MAX_BINS = 5
MIN_SUPPORT = 0.02
MIN_CONFIDENCE = 0.40
MIN_LIFT = 1.00
ECLAT_MAX_COMBINATION = 3
GROUP_RARE_CATEGORIES = True
MIN_CATEGORY_FREQUENCY = 0.005

DATASET_SLUG = "sadiajavedd/social-media-user-activity-dataset"
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 5. Descargar o cargar dataset

Para Kaggle necesitas subir `kaggle.json`. Si ya tienes el CSV, define `CSV_PATH` manualmente.

In [ ]:
from google.colab import files

if not Path("/root/.kaggle/kaggle.json").exists():
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
        Path("/root/.kaggle/kaggle.json").write_bytes(uploaded["kaggle.json"])
        !chmod 600 /root/.kaggle/kaggle.json

In [ ]:
%%capture
!kaggle datasets download -d {DATASET_SLUG} -p {DATA_DIR} --force

In [ ]:
for zip_path in DATA_DIR.glob("*.zip"):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

csv_files = sorted(DATA_DIR.glob("*.csv"))
CSV_PATH = csv_files[0] if csv_files else Path("/content/data/social_media_user_activity.csv")
CSV_PATH

## 6. Carga, normalizacion y EDA inicial

In [ ]:
df_raw = load_dataset(CSV_PATH)
df = normalize_column_names(df_raw)

print(f"Dataset original: {df_raw.shape}")
print(f"Dataset con columnas normalizadas: {df.shape}")
df.head()

In [ ]:
column_analysis = analyze_column_cardinality(df)
column_analysis

In [ ]:
display(report_missing_values(df).head(20))
report_duplicates(df)

## 7. Eliminacion de columnas que no aportan

La lista es editable. Se sugiere eliminar IDs, columnas constantes, cardinalidad extrema o demasiados nulos.

In [ ]:
suggested_drop_columns = suggest_columns_to_drop(df, target_column=TARGET_COLUMN)
suggested_drop_columns

In [ ]:
# Edita esta lista si quieres conservar o eliminar columnas adicionales.
columns_to_drop = suggested_drop_columns.copy()

df_reduced = drop_columns(df, columns_to_drop)
print(f"Columnas antes: {df.shape[1]}")
print(f"Columnas despues: {df_reduced.shape[1]}")
summarize_columns(df_reduced)

## 8. Seleccion final de columnas

Se propone una lista basada en la documentacion del proyecto, pero puedes editarla.

In [ ]:
suggested_columns = default_analysis_columns(df_reduced)
if TARGET_COLUMN in df_reduced.columns and TARGET_COLUMN not in suggested_columns:
    suggested_columns.append(TARGET_COLUMN)

selected_columns = suggested_columns if suggested_columns else df_reduced.columns.tolist()
df_selected = df_reduced[selected_columns].copy()
selected_columns

## 9. Limpieza, normalizacion de categorias y muestra de prueba

In [ ]:
df_clean = clean_selected_data(df_selected, numeric_strategy="median", categorical_strategy="unknown")
df_clean = normalize_categorical_values(df_clean)
df_work = sample_dataframe(df_clean, use_sample=USE_SAMPLE, sample_size=SAMPLE_SIZE, random_state=RANDOM_STATE)

print(f"Dataset limpio: {df_clean.shape}")
print(f"Dataset usado en esta prueba: {df_work.shape}")
df_work.head()

In [ ]:
if GROUP_RARE_CATEGORIES:
    categorical_columns = split_column_types(df_work)["categorical"]
    df_work = reduce_rare_categories(
        df_work,
        categorical_columns=categorical_columns,
        min_frequency=MIN_CATEGORY_FREQUENCY,
    )

summarize_columns(df_work)

## 10. Discretizacion y matriz binaria

Las variables numericas se separan por cuantiles en maximo `MAX_BINS` intervalos. Para prueba usamos menos bins; para ejecucion completa se puede subir a 10.

In [ ]:
df_discretized = discretize_numeric_columns(df_work, max_bins=MAX_BINS)
binary_matrix = create_binary_matrix(df_discretized)
validate_binary_matrix(binary_matrix)

## 11. Ejecutar Apriori, FP-Growth y Eclat

La comparacion no usa runtime. El criterio se basa en itemsets, reglas, reglas relacionadas con felicidad, soporte, confianza y lift.

In [ ]:
results = run_all_algorithms(
    binary_df=binary_matrix,
    min_support=MIN_SUPPORT,
    min_confidence=MIN_CONFIDENCE,
    min_lift=MIN_LIFT,
    target_column=TARGET_COLUMN,
    eclat_max_combination=ECLAT_MAX_COMBINATION,
)

comparison = summarize_algorithm_results(results)
comparison

In [ ]:
best_algorithm = choose_best_algorithm(comparison)
print(f"Mejor algoritmo segun calidad de reglas: {best_algorithm}")

## 12. Reglas asociadas a felicidad

In [ ]:
for algorithm, result in results.items():
    print(f"\n=== {algorithm.upper()} ===")
    readable_rules = rules_to_readable(result["happiness_rules"])
    display(readable_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(15))

## 13. Exportar resultados

In [ ]:
OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUT_DIR / "algorithm_comparison.csv", index=False)

for algorithm, result in results.items():
    result["itemsets"].to_csv(OUTPUT_DIR / f"{algorithm}_itemsets.csv", index=False)
    rules_to_readable(result["rules"]).to_csv(OUTPUT_DIR / f"{algorithm}_rules.csv", index=False)
    rules_to_readable(result["happiness_rules"]).to_csv(OUTPUT_DIR / f"{algorithm}_happiness_rules.csv", index=False)

sorted(OUTPUT_DIR.glob("*.csv"))

## 14. Ejecucion completa opcional

Cuando la prueba con muestra funcione, cambia estos parametros y vuelve a ejecutar desde la limpieza:

```python
USE_SAMPLE = False
MAX_BINS = 10
MIN_SUPPORT = 0.01
```

Si la matriz binaria queda demasiado grande para Colab, aumenta `MIN_SUPPORT` o reduce columnas/categorias antes de ejecutar los tres algoritmos.